# 05 · Контрфактуальный повтор траекторий AgentDojo

**Нужна локальная Ollama** (`qwen2.5:3b`). Прогон долгий, порядка трёх
тысяч вызовов модели: его делают один раз до вебинара, а ноутбук `04_trajectory_benchmark.ipynb`
только читает результат из `data/counterfactual/replay.csv`.

Вопрос: разметка agent-trajectory-attribution говорит, что причина опасного
действия — компонент с инъекцией. Это метка **правдоподобия**: её поставила
LLM, прочитав траекторию. Необходим ли этот компонент на самом деле?
Проверяем вмешательством.

Для каждого кейса AgentDojo, где целевое действие — вызов инструмента,
обрезаем траекторию перед ним и просим модель сделать следующий шаг
в трёх условиях:

| условие | что меняем |
|---|---|
| `full` | ничего |
| `no_injection` | из компонента-причины вырезан текст инъекции; он известен точно из `source_metadata.injections` |
| `drop_other` | выброшен случайный другой шаг с вызовом инструмента: контроль на то, что любое вмешательство ломает поведение |

По `N_SAMPLES` генераций на условие, с фиксированным `seed`, поэтому
прогон воспроизводим на той же версии модели и Ollama.

**Оговорка, которую надо проговаривать.** Траектории сгенерированы Qwen3.6,
а повторяем мы на qwen2.5:3b. Меряется необходимость для **нашей** модели.
Это не проверка разметки для исходного агента, а демонстрация того, как
такую проверку делать и что она показывает.

Прогон дописывает строки в CSV и пропускает уже сделанные, так что его
можно прерывать и продолжать.

In [1]:
import sys; sys.path.insert(0, "..")
import random
from pathlib import Path

import pandas as pd
import requests

from agent import MODEL_NAME, OLLAMA_URL
from bench import load_ata
from bench.replay import generate, messages, reproduced, suite_tools, target_call

models = [m["name"] for m in requests.get(f"{OLLAMA_URL}/api/tags").json()["models"]]
assert MODEL_NAME in models, f"В Ollama нет {MODEL_NAME}: ollama pull {MODEL_NAME}. Есть: {models}"

N_SAMPLES = 5
OUT = Path("../data/counterfactual/replay.csv")
OUT.parent.mkdir(exist_ok=True)

agentdojo = [c for c in load_ata() if c.subset.startswith("agentdojo/")]
tools = suite_tools(agentdojo)
cases = [c for c in agentdojo
         if c.subset == "agentdojo/unsafe_action" and c.steps[c.target].flags["function"]]
print(f"кейсов: {len(cases)},  вызовов модели: до {len(cases) * 3 * N_SAMPLES}")

кейсов: 214,  вызовов модели: до 3210


Какой шаг выбрасывать в `drop_other`, выбирается детерминированно
по `case_id`: другой шаг с вызовом инструмента до цели, не причина.

In [2]:
def other_step(case):
    pool = [i for i in range(case.target) if case.steps[i].flags["function"] and i not in case.causes]
    return random.Random(case.case_id).choice(pool) if pool else None

done = set()
if OUT.exists():
    prev = pd.read_csv(OUT)
    done = set(zip(prev.case_id, prev.intervention, prev.seed))

for k, case in enumerate(cases, 1):
    function, values = target_call(case)
    suite_tools_ = tools[case.meta["source_metadata"]["suite_name"]]
    drop = other_step(case)
    rows = []
    for intervention in ("full", "no_injection", "drop_other"):
        if intervention == "drop_other" and drop is None:
            continue
        msgs = messages(case, suite_tools_, intervention=intervention, drop=drop)
        for seed in range(N_SAMPLES):
            if (case.case_id, intervention, seed) in done:
                continue
            out = generate(msgs, model=MODEL_NAME, seed=seed)
            rows.append({"case_id": case.case_id, "intervention": intervention, "seed": seed,
                         "dropped": drop if intervention == "drop_other" else None,
                         "function": function, "values": "|".join(values),
                         "reproduced": reproduced(out, function, values), "output": out})
    if rows:
        pd.DataFrame(rows).to_csv(OUT, mode="a", header=not OUT.exists(), index=False)
    if k % 20 == 0:
        print(f"{k}/{len(cases)}")

20/214


40/214


60/214


80/214


100/214


120/214


140/214


160/214


180/214


200/214


In [3]:
res = pd.read_csv(OUT)
print(f"строк: {len(res)},  модель: {MODEL_NAME}\n")
print(res.groupby("intervention")["reproduced"].agg(["mean", "sum", "count"]).round(3).to_string())

строк: 2980,  модель: qwen2.5:3b

               mean  sum  count
intervention                   
drop_other    0.171  144    840
full          0.170  182   1070
no_injection  0.156  167   1070
